# Exploratory Data Analysis

Goal: understand both source datasets before building the pipeline.
Documents data quality issues that inform Silver layer design decisions.

In [0]:
import json
import csv
import re
import collections
import statistics

AIRBNB_PATH = "../data/airbnb.csv"
RENTALS_PATH = "../data/rentals.json"
POSTCODES_PATH = "../data/geo/post_codes.geojson"
AREAS_PATH = "../data/geo/amsterdam_areas.geojson"

## 1. Airbnb dataset

In [0]:
with open(AIRBNB_PATH) as f:
    airbnb = list(csv.DictReader(f))

print(f"Rows: {len(airbnb)}")
print(f"Columns: {list(airbnb[0].keys())}")

In [0]:
# Missing value audit
for col in airbnb[0].keys():
    missing = sum(1 for r in airbnb if not r[col].strip())
    pct = missing / len(airbnb) * 100
    print(f"{col:30s}  missing: {missing:4d}  ({pct:.1f}%)")

In [0]:
# Price distribution
prices = [float(r['price']) for r in airbnb if r['price']]
prices_sorted = sorted(prices)
n = len(prices)
q1 = prices_sorted[n // 4]
q3 = prices_sorted[3 * n // 4]
iqr = q3 - q1
upper_fence = q3 + 1.5 * iqr

print(f"Count : {n}")
print(f"Min   : {min(prices):.0f}")
print(f"Q1    : {q1:.0f}")
print(f"Median: {statistics.median(prices):.0f}")
print(f"Q3    : {q3:.0f}")
print(f"Max   : {max(prices):.0f}")
print(f"IQR   : {iqr:.0f}")
print(f"Upper fence (Q3 + 1.5*IQR): {upper_fence:.0f}")
outliers = sum(1 for p in prices if p > upper_fence)
print(f"Outliers above fence: {outliers} ({outliers/n*100:.1f}%)")

In [0]:
# Room type distribution
room_types = collections.Counter(r['room_type'] for r in airbnb)
for rt, count in room_types.most_common():
    print(f"{rt:25s} {count:5d}  ({count/len(airbnb)*100:.1f}%)")

In [0]:
# Zipcode coverage
has_zip = [r for r in airbnb if r['zipcode'].strip()]
no_zip  = [r for r in airbnb if not r['zipcode'].strip()]
print(f"Has zipcode : {len(has_zip)} ({len(has_zip)/len(airbnb)*100:.1f}%)")
print(f"Missing zipcode: {len(no_zip)} ({len(no_zip)/len(airbnb)*100:.1f}%)")
print()
# All rows with missing zipcode still have lat/lon — needed for backfill
has_coords = sum(1 for r in no_zip if r['latitude'] and r['longitude'])
print(f"Of missing-zip rows, have lat/lon: {has_coords}/{len(no_zip)}")
print()
print("Top 10 zipcodes:")
for zc, cnt in collections.Counter(r['zipcode'] for r in has_zip).most_common(10):
    print(f"  {zc}  {cnt}")

### Airbnb data quality summary

| Issue | Detail | Silver action |
|---|---|---|
| Missing zipcode | 22.7% of rows | Backfill via point-in-polygon using `post_codes.geojson` |
| Price outliers | Max €9,000; IQR upper fence ~€X | Cap at IQR upper fence or flag as outlier |
| `bedrooms` nulls | ~X% | Acceptable; not used in revenue calc |
| `zipcode` dtype | Stored as float in some rows (e.g. `1053.0`) | Cast to 4-char string |
| `review_scores_value` nulls | ~X% | Acceptable; optional enrichment |

## 2. Rentals (Kamernet) dataset

In [0]:
with open(RENTALS_PATH) as f:
    rentals = json.load(f)

print(f"Total records: {len(rentals)}")
print(f"Keys: {list(rentals[0].keys())}")

In [0]:
# City distribution — why we filter to Amsterdam in Silver
cities = collections.Counter(r.get('city', '') for r in rentals)
print("Top 15 cities:")
for city, cnt in cities.most_common(15):
    print(f"  {city:20s} {cnt:5d}  ({cnt/len(rentals)*100:.1f}%)")

amsterdam = [r for r in rentals if r.get('city') == 'Amsterdam']
print(f"\nAmsterdam records: {len(amsterdam)} ({len(amsterdam)/len(rentals)*100:.1f}% of total)")

In [0]:
# Rent field — raw string needs parsing
print("Raw rent samples:")
for r in rentals[:5]:
    print(f"  {repr(r.get('rent', ''))}")

def parse_eur(raw):
    """Extract numeric value from a raw euro string like '\n€ 750\n'."""
    if isinstance(raw, list):
        raw = raw[0] if raw else ''
    m = re.search(r'[\d.]+', str(raw).replace(',', ''))
    return float(m.group()) if m else None

rents = [parse_eur(r.get('rent')) for r in amsterdam]
rents = [x for x in rents if x]
rents_sorted = sorted(rents)
n = len(rents)
print(f"\nAmsterdam rent distribution (n={n}):")
print(f"  Min   : {min(rents):.0f}")
print(f"  Median: {statistics.median(rents):.0f}")
print(f"  Mean  : {sum(rents)/n:.0f}")
print(f"  Max   : {max(rents):.0f}")

In [0]:
# Property type distribution (Amsterdam)
ptypes = collections.Counter(r.get('propertyType', '') for r in amsterdam)
print("Property types (Amsterdam):")
for pt, cnt in ptypes.most_common():
    print(f"  {pt:25s} {cnt:5d}  ({cnt/len(amsterdam)*100:.1f}%)")

In [0]:
# Area distribution
def parse_sqm(raw):
    if isinstance(raw, list):
        raw = raw[0] if raw else ''
    m = re.search(r'(\d+)', str(raw))
    return int(m.group()) if m else None

areas = [parse_sqm(r.get('areaSqm')) for r in amsterdam]
areas = [x for x in areas if x]
print(f"Area distribution (Amsterdam, n={len(areas)}):")
print(f"  Min   : {min(areas)} m²")
print(f"  Median: {statistics.median(areas)} m²")
print(f"  Mean  : {sum(areas)/len(areas):.0f} m²")
print(f"  Max   : {max(areas)} m²")

In [0]:
# List fields that need flattening (stored as single-element lists in JSON)
list_fields = [k for k, v in rentals[0].items() if isinstance(v, list)]
print("Fields stored as lists (need flattening):")
for f in list_fields:
    sample = rentals[0][f]
    print(f"  {f:25s} sample: {sample}")

In [0]:
# Postal code coverage in Amsterdam subset
has_pc = sum(1 for r in amsterdam if r.get('postalCode', '').strip())
print(f"Postal code coverage (Amsterdam): {has_pc}/{len(amsterdam)} ({has_pc/len(amsterdam)*100:.1f}%)")

# PC4 extraction (first 4 digits)
pc4s = [str(r.get('postalCode', ''))[:4] for r in amsterdam if r.get('postalCode', '').strip()]
pc4_counts = collections.Counter(pc4s)
print("\nTop 10 PC4 postcodes (Amsterdam rentals):")
for pc, cnt in pc4_counts.most_common(10):
    print(f"  {pc}  {cnt}")

### Rentals data quality summary

| Issue | Detail | Silver action |
|---|---|---|
| NL-wide dataset | Only 17% of records are Amsterdam | Filter `city == 'Amsterdam'` in Silver |
| `rent` raw string | `'\n€ 750\n'` format | Parse with regex, cast to float |
| `areaSqm` raw string | `'14 m2'` format | Extract integer with regex |
| `deposit` raw string | Same format as rent | Parse with regex |
| List-wrapped fields | `_id`, `crawledAt`, etc. are `['value']` | Flatten to scalar |
| `additionalCostsRaw` | Raw string | Parse to float; add to rent for total cost |
| Data age | Crawled ~2019 | Note in docs; use as-is since no fresher data |

## 3. Geographic data

In [0]:
with open(POSTCODES_PATH) as f:
    postcodes_geo = json.load(f)

pc_features = postcodes_geo['features']
print(f"Postcode polygons: {len(pc_features)}")
print(f"Properties: {list(pc_features[0]['properties'].keys())}")

# PC4 codes that cover Amsterdam
ams_pcs = [
    ft['properties']['pc4_code']
    for ft in pc_features
    if ft['properties'].get('gem_name') == 'Amsterdam'
]
print(f"\nAmsterdam PC4 codes in GeoJSON: {len(ams_pcs)}")
print(f"  Range: {min(ams_pcs)} – {max(ams_pcs)}")

In [0]:
with open(AREAS_PATH) as f:
    areas_geo = json.load(f)

area_features = areas_geo['features']
print(f"Neighbourhood polygons: {len(area_features)}")
print(f"Properties: {list(area_features[0]['properties'].keys())}")
print("\nSample neighbourhoods:")
for ft in area_features[:5]:
    p = ft['properties']
    print(f"  {p['Buurtcombinatie_code']}  {p['Buurtcombinatie']}")

## 4. Cross-dataset alignment check

In [0]:
# Which Airbnb zipcodes match the GeoJSON PC4 codes?
geo_pc4s = {ft['properties']['pc4_code'] for ft in pc_features}
airbnb_zips = {r['zipcode'].strip().split('.')[0] for r in airbnb if r['zipcode'].strip()}

matched = airbnb_zips & geo_pc4s
unmatched = airbnb_zips - geo_pc4s
print(f"Airbnb unique zipcodes     : {len(airbnb_zips)}")
print(f"  In GeoJSON               : {len(matched)}")
print(f"  Not in GeoJSON           : {len(unmatched)}")
if unmatched:
    print(f"  Unmatched values: {sorted(unmatched)[:10]}")

In [0]:
# Which rentals PC4s match Airbnb PC4s? Overlap = postcodes we can compare.
rental_pc4s = {str(r.get('postalCode', ''))[:4] for r in amsterdam if r.get('postalCode', '').strip()}
overlap = airbnb_zips & rental_pc4s
print(f"Amsterdam rental unique PC4s : {len(rental_pc4s)}")
print(f"Airbnb unique PC4s           : {len(airbnb_zips)}")
print(f"Overlap (comparable)         : {len(overlap)}")

## 5. Key findings & pipeline design decisions

| Finding | Pipeline decision |
|---|---|
| 22.7% Airbnb rows lack zipcode but have lat/lon | Silver: point-in-polygon backfill using `post_codes.geojson` |
| Rentals dataset covers all NL | Silver: filter `city == 'Amsterdam'` |
| Rent/deposit/area are raw strings | Silver: regex parsing to numeric |
| Several JSON fields are single-element lists | Silver: flatten before writing |
| Airbnb price in €/night; rentals in €/month | Gold: normalise Airbnb to monthly (×365×0.66/12) |
| Rentals data from 2019 | Document as limitation; no fresher source available |
| Amsterdam has ~60 PC4 postcodes in GeoJSON | Gold: aggregate both datasets at PC4 level |